# RAG Chatbot — Reglamento Beca 18 (PRONABEC)
## Resolución Directoral Ejecutiva N.° 033-2026-MINEDU/VMGI-PRONABEC

Pipeline: PDF → extracción → chunks → embeddings → ChromaDB → búsqueda semántica → respuesta fundamentada

---
## Step 0 — Setup

In [ ]:
# Instalar dependencias
!pip install pypdf tiktoken langchain-text-splitters google-genai chromadb ipywidgets tqdm python-dotenv -q

In [30]:
import os
import time
import re
import random
from pathlib import Path

import pypdf
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions
import google.genai as genai
from google.genai import types
import ipywidgets as widgets
from IPython.display import display, clear_output
from tqdm import tqdm
from dotenv import load_dotenv

# Cargar API key desde .env
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    raise ValueError('GEMINI_API_KEY no encontrada. Verifica tu archivo .env')

# Configurar cliente Gemini
client = genai.Client(api_key=GEMINI_API_KEY)

print('=== Versiones de librerías ===')
print(f'pypdf            : {pypdf.__version__}')
print(f'tiktoken         : {tiktoken.__version__}')
print(f'chromadb         : {chromadb.__version__}')
print(f'google-genai     : {genai.__version__}')
print(f'ipywidgets       : {widgets.__version__}')
print('\n✓ API key cargada correctamente')
print('✓ Entorno listo')

=== Versiones de librerías ===
pypdf            : 6.11.0
tiktoken         : 0.12.0
chromadb         : 1.5.9
google-genai     : 1.68.0
ipywidgets       : 7.7.1

✓ API key cargada correctamente
✓ Entorno listo


---
## Step 1 — PDF Text Extraction

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('data', exist_ok=True)

import shutil
shutil.copy('/content/drive/MyDrive/Colab Notebooks/data/beca18_reglamento.pdf', 'data/beca18_reglamento.pdf')
print('✓ PDF copiado correctamente')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ PDF copiado correctamente


In [ ]:
PDF_PATH = 'data/beca18_reglamento.pdf'

def extract_text_from_pdf(pdf_path):
    """
    Extrae texto página por página insertando marcadores [PAGE N].
    Aplica limpieza ligera: colapsa espacios, elimina saltos de línea aislados.
    """
    reader = pypdf.PdfReader(pdf_path)
    full_text = []

    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''

        # Limpieza ligera
        text = re.sub(r'[ \t]+', ' ', text)           # colapsar espacios múltiples
        text = re.sub(r'\n{3,}', '\n\n', text)        # máximo 2 saltos consecutivos
        text = re.sub(r' \n ', ' ', text)              # eliminar saltos aislados
        text = text.strip()

        full_text.append(f'[PAGE {i}]\n{text}')

    return '\n\n'.join(full_text)

print('Extrayendo texto del PDF...')
cleaned_text = extract_text_from_pdf(PDF_PATH)

total_chars = len(cleaned_text)
total_words = len(cleaned_text.split())
total_pages = cleaned_text.count('[PAGE ')

print(f'\n=== Extracción completada ===')
print(f'  Páginas extraídas : {total_pages}')
print(f'  Total caracteres  : {total_chars:,}')
print(f'  Total palabras    : {total_words:,}')
print(f'\nPrimeros 500 caracteres:')
print(cleaned_text[:500])

Extrayendo texto del PDF...

=== Extracción completada ===
  Páginas extraídas : 138
  Total caracteres  : 372,595
  Total palabras    : 55,202

Primeros 500 caracteres:
[PAGE 1]
Resolución Directoral Ejecutiva 
Nº 033-2026-MINEDU/VMGI-PRONABEC 
 Lima, 24 de febrero de 2026 
VISTOS: 
El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por 
la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y 
Bienestar; el Informe N° 042-2026-MINEDU/VMGI-PRONABEC-OPP de la Oficina de 
Planeamiento y Presupuesto; el Informe N ° 048-2026-MINEDU/VMGI-PRONABEC-OAJ 
de la Oficina de Asesoría Jurídica, y; 
CONSIDERANDO: 
Que, la Ley N° 29837 c


---
## Step 2 — Tokenization and Chunking

In [ ]:
# Contar tokens con tiktoken
encoding = tiktoken.get_encoding('cl100k_base')
tokens = encoding.encode(cleaned_text)
total_tokens = len(tokens)

print(f'=== Conteo de Tokens ===')
print(f'  Total tokens (cl100k_base): {total_tokens:,}')

=== Conteo de Tokens ===
  Total tokens (cl100k_base): 109,751


### Justificación del tamaño de chunk

El modelo `gemini-embedding-001` tiene un límite de **8,192 tokens** por request de embedding, lo que nos da bastante margen para trabajar. Elegí un **chunk_size de 400 tokens** con un **overlap de 60 tokens** por varias razones prácticas.

Primero, el reglamento de Beca 18 está estructurado en artículos cortos y listas de requisitos, así que 400 tokens es suficiente para capturar entre 1 y 3 artículos completos sin perder contexto. Usar chunks más grandes haría que el retrieval sea menos preciso, porque mezclaría temas distintos en un mismo fragmento.

Segundo, 400 tokens representa apenas el 4.9% del límite máximo del modelo, lo que nos da un margen de seguridad enorme y evita cualquier error por overflow.

Tercero, el overlap de 60 tokens (15% del chunk) es clave para no perder información en los bordes: si un artículo importante está partido entre dos chunks, el overlap garantiza que ambos lo capturen parcialmente y que el retrieval pueda encontrarlo desde cualquier ángulo.

En resumen, este tamaño de chunk balancea granularidad, precisión de búsqueda y eficiencia de uso de la API.

In [ ]:
# Chunking con RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 400,
    chunk_overlap = 60,
    separators    = ['\n\n', '\n', '. ', ' '],
    length_function = lambda text: len(encoding.encode(text))
)

raw_chunks = splitter.split_text(cleaned_text)

# Adjuntar metadata a cada chunk
def get_page_number(chunk_text):
    """Extrae el número de página más reciente del chunk."""
    matches = re.findall(r'\[PAGE (\d+)\]', chunk_text)
    return matches[-1] if matches else 'unknown'

chunks = []
for i, chunk_text in enumerate(raw_chunks):
    page = get_page_number(chunk_text)
    chunks.append({
        'id'      : f'chunk_{i:04d}',
        'text'    : chunk_text,
        'metadata': {
            'document': 'RDE-033-2026-MINEDU-PRONABEC',
            'topic'   : 'Beca 18 Reglamento',
            'language': 'es',
            'page'    : page,
            'chunk_id': i
        }
    })

avg_len = sum(len(c['text']) for c in chunks) / len(chunks)
print(f'=== Chunking completado ===')
print(f'  Total chunks         : {len(chunks)}')
print(f'  Longitud media (chars): {avg_len:.0f}')
print(f'\nEjemplo chunk 0:')
print(chunks[0]['text'][:300])
print(f'  Metadata: {chunks[0]["metadata"]}')

=== Chunking completado ===
  Total chunks         : 385
  Longitud media (chars): 1062

Ejemplo chunk 0:
[PAGE 1]
Resolución Directoral Ejecutiva 
Nº 033-2026-MINEDU/VMGI-PRONABEC 
 Lima, 24 de febrero de 2026 
VISTOS: 
El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por 
la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y 
Bienestar; el Informe N° 042-202
  Metadata: {'document': 'RDE-033-2026-MINEDU-PRONABEC', 'topic': 'Beca 18 Reglamento', 'language': 'es', 'page': '1', 'chunk_id': 0}


---
## Step 3 — Embeddings

In [ ]:
def embed_documents(texts, max_retries=5):
    """
    Genera embeddings para una lista de textos (documentos a indexar).
    Usa task_type RETRIEVAL_DOCUMENT con backoff exponencial.
    """
    embeddings = []
    for text in texts:
        for attempt in range(max_retries):
            try:
                result = client.models.embed_content(
                    model   = 'gemini-embedding-001',
                    contents = text,
                    config  = types.EmbedContentConfig(
                        task_type = 'RETRIEVAL_DOCUMENT'
                    )
                )
                embeddings.append(result.embeddings[0].values)
                break
            except Exception as e:
                if attempt < max_retries - 1:
                    wait = (2 ** attempt) + random.uniform(0, 1)
                    time.sleep(wait)
                else:
                    raise e
    return embeddings


def embed_query(text, max_retries=5):
    """
    Genera embedding para una query de búsqueda.
    Usa task_type RETRIEVAL_QUERY.
    """
    for attempt in range(max_retries):
        try:
            result = client.models.embed_content(
                model   = 'gemini-embedding-001',
                contents = text,
                config  = types.EmbedContentConfig(
                    task_type = 'RETRIEVAL_QUERY'
                )
            )
            return result.embeddings[0].values
        except Exception as e:
            if attempt < max_retries - 1:
                wait = (2 ** attempt) + random.uniform(0, 1)
                time.sleep(wait)
            else:
                raise e

# Test rápido
test_emb = embed_query('¿Qué es Beca 18?')
print(f'✓ Embedding de prueba generado')
print(f'  Dimensiones: {len(test_emb)} (modelo gemini-embedding-001)')

✓ Embedding de prueba generado
  Dimensiones: 3072 (modelo gemini-embedding-001)


---
## Step 4 — Vector Database (ChromaDB)

In [ ]:
CHROMA_PATH      = 'chroma_db_beca18'
COLLECTION_NAME  = 'beca18_reglamento'
BATCH_SIZE       = 5   # pequeño para respetar rate limit free tier

# Cliente persistente
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Crear o cargar colección con distancia coseno
collection = chroma_client.get_or_create_collection(
    name     = COLLECTION_NAME,
    metadata = {'hnsw:space': 'cosine'}
)

existing_count = collection.count()
print(f'Documentos existentes en la colección: {existing_count}')

if existing_count > 0:
    print('✓ Colección ya poblada — saltando embedding (idempotente)')
else:
    print(f'Indexando {len(chunks)} chunks en lotes de {BATCH_SIZE}...')
    print('(Esto puede tomar varios minutos por el rate limit del tier gratuito)')

    for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc='Indexando'):
        batch = chunks[i:i + BATCH_SIZE]
        texts     = [c['text']     for c in batch]
        ids       = [c['id']       for c in batch]
        metadatas = [c['metadata'] for c in batch]

        embeds = embed_documents(texts)

        collection.add(
            ids        = ids,
            documents  = texts,
            embeddings = embeds,
            metadatas  = metadatas
        )

        # Pausa para respetar rate limit (~60 req/min)
        time.sleep(1.5)

print(f'\n✓ Total documentos en ChromaDB: {collection.count()}')

Documentos existentes en la colección: 385
✓ Colección ya poblada — saltando embedding (idempotente)

✓ Total documentos en ChromaDB: 385


---
## Step 5 — Semantic Search

In [ ]:
def semantic_search(question, k=5):
    """
    Busca los k chunks más relevantes para la pregunta.
    Retorna lista de dicts con text, metadata y distance.
    """
    query_embedding = embed_query(question)

    results = collection.query(
        query_embeddings = [query_embedding],
        n_results        = k,
        include          = ['documents', 'metadatas', 'distances']
    )

    output = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        output.append({
            'text'    : doc,
            'metadata': meta,
            'distance': dist
        })

    return output

# Test con pregunta de muestra
test_question = '¿Cuáles son los requisitos para postular a Beca 18?'
print(f'Pregunta de prueba: {test_question}\n')

results = semantic_search(test_question, k=3)
for i, r in enumerate(results, 1):
    print(f'--- Resultado {i} ---')
    print(f'  Distancia : {r["distance"]:.4f}')
    print(f'  Página    : {r["metadata"]["page"]}')
    print(f'  Texto     : {r["text"][:200]}...')
    print()

Pregunta de prueba: ¿Cuáles son los requisitos para postular a Beca 18?

--- Resultado 1 ---
  Distancia : 0.1940
  Página    : 102
  Texto     : [PAGE 102]
16 
N° REQUISITO DOCUMENTO DE ACREDITACIÓN o FORMA 
DE ACREDITACIÓN 
documento oficial de la IES donde se acredite 
que mantiene dicha vacante. 
2 
Haber culminado el nivel 
secundario de l...

--- Resultado 2 ---
  Distancia : 0.1962
  Página    : unknown
  Texto     : éstos. 
En el caso de estudios de educación básica 
realizados en el extranjero, deberá cargar la 
Resolución de reconocimiento de los estudios 
realizados en el extranjero del Ministerio de 
Educació...

--- Resultado 3 ---
  Distancia : 0.2101
  Página    : 104
  Texto     : [PAGE 104]
18 
N° REQUISITO DOCUMENTO DE ACREDITACIÓN o FORMA 
DE ACREDITACIÓN 
Para la Beca REPARED: 
acreditar nota mínima 12.00 en 
los dos últimos grados 
concluidos de secundaria de 
EBR o EBA o ...



---
## Step 6 — Grounded Generation

In [ ]:
SYSTEM_PROMPT = """Eres un asistente especializado en el Reglamento de Beca 18 de PRONABEC (Resolución Directoral Ejecutiva N.° 033-2026-MINEDU/VMGI-PRONABEC).

REGLAS ESTRICTAS:
1. Responde ÚNICAMENTE basándote en el contexto proporcionado. No uses conocimiento propio.
2. Cuando uses información de una página específica, cítala así: (Página N).
3. Si el contexto no contiene información suficiente para responder, di exactamente: "El documento no contiene información sobre este tema."
4. No inventes, no supongas, no extrapoles información fuera del contexto.
5. Responde en español, de forma clara y estructurada."""


def answer_with_context(question, k=5):
    """
    Recupera contexto relevante y genera respuesta fundamentada con Gemini.
    """
    # Recuperar chunks relevantes
    retrieved = semantic_search(question, k=k)

    # Construir contexto
    context_parts = []
    for r in retrieved:
        page = r['metadata'].get('page', 'N/A')
        context_parts.append(f'[Página {page}]\n{r["text"]}')

    context = '\n\n---\n\n'.join(context_parts)

    # Prompt con contexto
    user_message = f"""CONTEXTO DEL REGLAMENTO:
{context}

PREGUNTA: {question}

Responde basándote exclusivamente en el contexto anterior."""

    response = client.models.generate_content(
        model    = 'gemini-2.5-flash',
        contents = [
            types.Content(role='user', parts=[types.Part(text=user_message)])
        ],
        config = types.GenerateContentConfig(
            system_instruction = SYSTEM_PROMPT,
            temperature        = 0.1,
            max_output_tokens  = 1024
        )
    )

    return {
        'answer'   : response.text,
        'sources'  : retrieved
    }

print('✓ Función answer_with_context lista')

✓ Función answer_with_context lista


In [ ]:
# Test con 5 preguntas on-topic + 1 off-topic
questions = [
    ('ON-TOPIC',  '¿Cuáles son los requisitos de elegibilidad para postular a Beca 18?'),
    ('ON-TOPIC',  '¿Cuáles son las modalidades de la beca?'),
    ('ON-TOPIC',  '¿Qué financia la subvención de Beca 18?'),
    ('ON-TOPIC',  '¿Cuáles son las obligaciones del estudiante becario?'),
    ('ON-TOPIC',  '¿Cuál es el presupuesto total asignado para Beca 18 convocatoria 2026?'),
    ('OFF-TOPIC', '¿Cuál es la receta del ceviche peruano?'),
]

for q_type, question in questions:
    print(f'\n{"="*60}')
    print(f'[{q_type}] {question}')
    print('='*60)
    result = answer_with_context(question, k=10)
    print(result['answer'])
    time.sleep(2)


[ON-TOPIC] ¿Cuáles son los requisitos de elegibilidad para postular a Beca 18?
Para postular a Beca 18 Ordinaria, los requisitos de elegibilidad son los siguientes:

1.  **Edad:** Ser menor de 22 años a la fecha

[ON-TOPIC] ¿Cuáles son las modalidades de la beca?
Las modalidades de la beca son las siguientes:

*   Beca 18 (ordinaria) (Página 89)
*   Beca de Formación en Educación Intercultural Bilingüe (Beca EIB) (Página 89)
*   Beca para adolescentes con protección estatal (Beca Protección) (Página 89)
*   Beca para Comunidades Nativas Amazónicas (Beca CNA) (Página 89)
*   Beca para licenciados del Servicio Militar Voluntario (Beca FF.AA.) (Página 89)
*   Beca para pobladores residentes del valle de los ríos Apurímac, Ene y Mantaro (Beca VRAEM) (Página 89)
*   Beca para pobladores residentes en el Huallaga (Beca Huallaga) (Página 89)
*   Beca para Pueblo Afroperuano (Beca PA) (Página 89)
*   Beca para víctimas de la violencia habida en el país durante los años 1980 – 2000 (Beca REPAR

---
## Step 7 — Interactive Chat Interface

In [31]:
# Widgets
question_input = widgets.Text(
    placeholder = '¿Cuáles son los requisitos para postular a Beca 18?',
    description = 'Pregunta:',
    layout      = widgets.Layout(width='70%')
)

k_slider = widgets.IntSlider(
    value       = 5,
    min         = 1,
    max         = 10,
    step        = 1,
    description = 'k (chunks):',
    layout      = widgets.Layout(width='50%')
)

ask_button   = widgets.Button(description='Preguntar', button_style='primary')
clear_button = widgets.Button(description='Limpiar',   button_style='warning')

answer_output = widgets.Output()
sources_accordion = widgets.Accordion(children=[widgets.Output()])
sources_accordion.set_title(0, 'Fuentes recuperadas (click para expandir)')

def on_ask(b):
    question = question_input.value.strip()
    if not question:
        return

    with answer_output:
        clear_output()
        print('Buscando respuesta...')

    try:
        result = answer_with_context(question, k=k_slider.value)

        with answer_output:
            clear_output()
            print(f'Pregunta: {question}\n')
            print('Respuesta:')
            print('-' * 50)
            print(result['answer'])

        with sources_accordion.children[0]:
            clear_output()
            for i, src in enumerate(result['sources'], 1):
                print(f'\n[Fuente {i}] Página {src["metadata"]["page"]} | Distancia: {src["distance"]:.4f}')
                print('-' * 40)
                print(src['text'][:400])
                print()

    except Exception as e:
        with answer_output:
            clear_output()
            print(f'Error: {e}')

def on_clear(b):
    question_input.value = ''
    with answer_output:
        clear_output()
    with sources_accordion.children[0]:
        clear_output()

ask_button.on_click(on_ask)
clear_button.on_click(on_clear)

# Layout
ui = widgets.VBox([
    widgets.HTML('<h2>🎓 Chatbot — Reglamento Beca 18 PRONABEC</h2>'),
    question_input,
    k_slider,
    widgets.HBox([ask_button, clear_button]),
    widgets.HTML('<hr>'),
    answer_output,
    sources_accordion
])

display(ui)